# Notebook 02 — Treinamento: Benchmark de Mecanismos de Atenção

**Aula 02 — Desafios de Performance em NLP e Áudio**  
Disciplina: Engenharia de Machine Learning — FIAP PósTech

---

## Objetivos

1. Implementar e comparar mecanismos de atenção: Vanilla, Linear, Local e Tiled (Flash-style).
2. Medir latência e memória para diferentes comprimentos de sequência.
3. Demonstrar o KV-Cache para geração autoregressiva.
4. Treinar um preditor simples de latência para consolidar a análise.

## Fundamentação Teórica

Da seção *Flash Attention: IO-Awareness como Princípio de Design*:

> Flash Attention resolve [o problema da memória O(n²)] através de
> **tiling** e **recomputação**. O algoritmo divide Q, K, V em blocos
> que cabem na SRAM, computa atenção parcial para cada bloco, e
> acumula resultados incrementalmente.

Da *Taxonomia de Atenção Eficiente* (Tay et al., 2022):

| Abordagem | Tempo | Memória | Qualidade |
|-----------|-------|---------|----------|
| Vanilla | O(n²·d) | O(n²) | Baseline |
| Flash | O(n²·d) | **O(n)** | Exata |
| Linear | O(n·d²) | O(n·d) | ~90-95% |
| Local | O(n·w·d) | O(n·w) | Degradada p/ long-range |

In [ ]:
# Imports
import sys
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath(".."))

from src.model import (
    VanillaAttention,
    LinearAttention,
    LocalAttention,
    TiledAttention,
    KVCache,
    SimpleTransformerBlock,
)
from src.training import (
    benchmark_attention_mechanisms,
    benchmark_kv_cache,
    benchmark_transformer_block,
    train_latency_predictor,
)
from src.evaluation import PerformanceEvaluator
from src.data_preprocessing import DataPreprocessor
from src.utils import Timer, format_bytes, save_metrics

sns.set_theme(style="whitegrid")
np.random.seed(42)
print("Imports carregados com sucesso.")

## 1. Implementação: Self-Attention Vanilla

Implementação direta da fórmula de Vaswani et al. (2017):

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

O fator $\sqrt{d_k}$ é crucial para estabilidade numérica.

In [ ]:
# Demonstração do self-attention vanilla com sequência curta
d_model = 64
num_heads = 4
seq_len = 16
batch_size = 1

rng = np.random.default_rng(42)
X = rng.normal(0, 1, (batch_size, seq_len, d_model)).astype(np.float32)

attn = VanillaAttention(d_model, num_heads)
output, weights = attn.forward(X)

print(f"Entrada:  {X.shape}")
print(f"Saída:    {output.shape}")
print(f"Weights:  {weights.shape}  → (batch, heads, n, n) — matriz n×n!")
print(f"Memória attention matrix: {format_bytes(attn.memory_footprint_bytes(seq_len))}")

In [ ]:
# Visualizar heatmap de atenção (head 0)
evaluator = PerformanceEvaluator()
fig = evaluator.plot_attention_heatmap(
    weights[0],  # batch 0, todas as heads
    title=f"Attention Weights — Vanilla ({num_heads} heads, n={seq_len})",
    save=False
)
plt.show()

## 2. Comparação: Vanilla vs Linear vs Local vs Tiled

Conforme a *Tabela Comparativa: Abordagens de Atenção Eficiente*:

- **Vanilla**: O(n²·d) tempo, O(n²) memória — baseline
- **Linear** (Katharopoulos et al., 2020): O(n·d²) tempo, O(n·d) memória
- **Local** (Beltagy et al., 2020): O(n·w·d) tempo, O(n·w) memória
- **Tiled** (Dao et al., 2022): O(n²·d) tempo, **O(n) memória** — Flash-style

In [ ]:
# Verificar que todos produzem outputs do shape correto
d_model = 64
seq_len = 32
X = rng.normal(0, 1, (1, seq_len, d_model)).astype(np.float32)

models = {
    "Vanilla": VanillaAttention(d_model, num_heads=2),
    "Linear": LinearAttention(d_model, num_heads=2),
    "Local (w=16)": LocalAttention(d_model, window_size=16, num_heads=2),
    "Tiled (B=16)": TiledAttention(d_model, block_size=16),
}

for name, model in models.items():
    out, _ = model.forward(X)
    mem = model.memory_footprint_bytes(seq_len)
    print(f"{name:20s} → output {out.shape}, memória atenção: {format_bytes(mem)}")

In [ ]:
# Benchmark em múltiplos comprimentos de sequência
print("Executando benchmarks (pode levar 1-2 minutos)...")
results = benchmark_attention_mechanisms(
    seq_lengths=[32, 64, 128, 256],
    d_model=64,
    num_heads=2,
    batch_size=2,
    n_runs=3,
)

# Exibir resultados como tabela
df_results = evaluator.results_to_dataframe(results)
df_results

In [ ]:
# Gráficos comparativos (Tabela Comparativa da seção Saiba Mais)
fig = evaluator.plot_scaling_comparison(results, save=True)
plt.show()
print("Figura salva em outputs/figures/scaling_comparison.png")

## 3. Flash Attention: Tiling em Detalhe

Conforme Dao et al. (2022):

> O algoritmo divide as matrizes Q, K, V em blocos que cabem na SRAM,
> computa atenção parcial para cada bloco, e acumula resultados
> incrementalmente. A softmax é computada de forma **online** usando
> estatísticas correntes de máximo e soma para renormalização.

$$O_i = \sum_j \frac{\exp(Q_i K_j^T - m_i)}{\sum_{j'} \exp(Q_i K_{j'}^T - m_i)} V_j$$

In [ ]:
# Comparar saída do Tiled vs Vanilla para validar corretude
d_model = 64
seq_len = 64
X = rng.normal(0, 1, (1, seq_len, d_model)).astype(np.float32)

vanilla = VanillaAttention(d_model, num_heads=1)
tiled = TiledAttention(d_model, block_size=16)

# Usar mesmos pesos
tiled.W_q = vanilla.W_q.copy()
tiled.W_k = vanilla.W_k.copy()
tiled.W_v = vanilla.W_v.copy()

out_vanilla, _ = vanilla.forward(X)
out_tiled, _ = tiled.forward(X)

# Comparar numericamente
max_diff = np.max(np.abs(out_vanilla - out_tiled))
mean_diff = np.mean(np.abs(out_vanilla - out_tiled))

print(f"Max diferença absoluta:  {max_diff:.6e}")
print(f"Mean diferença absoluta: {mean_diff:.6e}")
print(f"\n→ Flash Attention produz resultado EXATO (diferença apenas por arredondamento)")

In [ ]:
# Comparar memória: Vanilla O(n²) vs Tiled O(n)
print(f"{'Seq Length':>10} | {'Vanilla':>12} | {'Tiled (B=32)':>12} | {'Razão':>8}")
print("-" * 52)
for n in [64, 128, 256, 512, 1024, 2048]:
    v = VanillaAttention(64, 1)
    t = TiledAttention(64, 32)
    mem_v = v.memory_footprint_bytes(n)
    mem_t = t.memory_footprint_bytes(n)
    print(f"{n:>10} | {format_bytes(mem_v):>12} | {format_bytes(mem_t):>12} | {mem_v/mem_t:>7.1f}×")

## 4. KV-Cache para Geração Autoregressiva

Conforme a seção *KV-Cache e Geração Autoregressiva*:

> Sem otimização, a geração resultaria em complexidade O(n³).
> O KV-Cache armazena K e V de tokens anteriores, reduzindo o
> custo marginal de cada token para O(t·d).

$$K_{cached} = [K_1; K_2; ...; K_t], \quad V_{cached} = [V_1; V_2; ...; V_t]$$

In [ ]:
# Demonstrar crescimento do KV-Cache
cache = KVCache(d_model=128, num_layers=4, num_heads=4)

print(f"{'Token':>5} | {'Cache Length':>12} | {'Memória Cache':>14}")
print("-" * 40)

for t in range(10):
    new_k = rng.normal(0, 1, (1, 1, 128)).astype(np.float32)
    new_v = rng.normal(0, 1, (1, 1, 128)).astype(np.float32)
    for layer in range(4):
        cache.update(layer, new_k, new_v)
    print(f"{t:>5} | {cache.current_length():>12} | {format_bytes(cache.memory_bytes()):>14}")

print(f"\n→ Memória cresce linearmente: O(n·L·H·d)")
print(f"  (Shazeer, 2019; Touvron et al., 2023)")

In [ ]:
# Benchmark completo do KV-Cache
print("Executando benchmark do KV-Cache...")
cache_data = benchmark_kv_cache(
    total_tokens=128, d_model=128, num_layers=4, num_heads=4
)

fig = evaluator.plot_kv_cache_growth(cache_data, save=True)
plt.show()
print("Figura salva em outputs/figures/kv_cache_growth.png")

## 5. Decomposição da Latência: Attention vs FFN

Conforme a seção *Decomposição da Latência em Transformers*:

> As camadas Feed-Forward Network (FFN), tipicamente com dimensão
> 4× maior que a hidden dimension, respondem por aproximadamente
> **60% dos FLOPs totais** em inference (Shoeybi et al., 2019).

In [ ]:
# Perfilar componentes do bloco Transformer
block_results = benchmark_transformer_block(
    seq_lengths=[32, 64, 128, 256],
    d_model=64,
    num_heads=2,
    batch_size=2,
)

df_block = pd.DataFrame(block_results)
print(df_block.to_string(index=False))

# Gráfico stacked bar
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_block))
ax.bar(x, df_block["attention_ms"], label="Attention", color="steelblue")
ax.bar(x, df_block["ffn_ms"], bottom=df_block["attention_ms"], label="FFN", color="coral")
ax.set_xticks(x)
ax.set_xticklabels(df_block["seq_length"])
ax.set_xlabel("Comprimento da Sequência")
ax.set_ylabel("Latência (ms)")
ax.set_title("Decomposição: Attention vs FFN\n(Shoeybi et al., 2019)")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Treinamento: Preditor de Latência

Treinamos uma regressão linear para prever latência a partir de
parâmetros de configuração (seq_length, d_model, num_heads, etc.),
consolidando a relação entre complexidade teórica e latência observada.

In [ ]:
# Preparar dados do dataset
preprocessor = DataPreprocessor(seed=42)
dataset_path = os.path.join("..", "data", "raw", "dataset.csv")

if not os.path.exists(dataset_path):
    preprocessor.generate_dataset(output_path=dataset_path)

df = preprocessor.load_data(dataset_path)
df = preprocessor.clean_data(df)
X, y = preprocessor.prepare_features(df)
splits = preprocessor.split_data(X, y)

print(f"Treino:     {splits['X_train'].shape}")
print(f"Validação:  {splits['X_val'].shape}")
print(f"Teste:      {splits['X_test'].shape}")

In [ ]:
# Treinar preditor de latência
print("Treinando preditor de latência...")
result = train_latency_predictor(
    splits["X_train"], splits["y_train"],
    splits["X_val"], splits["y_val"],
    learning_rate=0.001,
    n_epochs=200,
)

print(f"Loss final treino:     {result['final_train_loss']:.4f}")
print(f"Loss final validação:  {result['final_val_loss']:.4f}")

# Curvas de loss
fig = evaluator.plot_training_loss(
    result["train_losses"], result["val_losses"], save=True
)
plt.show()

In [ ]:
# Salvar métricas e modelo
metrics = {
    "final_train_loss": result["final_train_loss"],
    "final_val_loss": result["final_val_loss"],
}
metrics_path = save_metrics(metrics, name="training_metrics")
print(f"Métricas salvas em: {metrics_path}")

## Resumo

Neste notebook implementamos e comparamos:

1. **Vanilla Attention**: Complexidade O(n²) verificada empiricamente.
2. **Linear Attention**: Kernel trick reduz memória para O(n·d) (Katharopoulos et al., 2020).
3. **Local Attention**: Janela fixa O(n·w) (Beltagy et al., 2020).
4. **Tiled Attention**: Resultado exato com memória O(n) — princípio do Flash Attention (Dao et al., 2022).
5. **KV-Cache**: Crescimento linear de memória na geração autoregressiva.
6. **Decomposição de latência**: FFN domina para sequências curtas.

---

**Próximo notebook:** [03_avaliacao.ipynb](03_avaliacao.ipynb) — Avaliação completa e análise comparativa.